# E4 stress audit: incomplete feedback and irregular shifts
New disjoint seed range 20270101–20270124. 24 independent streams per regime, 36 tasks each, irregular A9/B13/A5/B9 vs random. One target example at prediction; exact output accuracy over other 80 inputs. After scoring, total feedback is 1 (no new examples), 4, or 81 examples. All methods receive identical evidence and candidate space. Feedback conditions share streams; do not pool as independent replicates. All methods see the same complete source truth table. No hidden regime labels. No hyperparameter changes. Test whether hard context reuse improves on change-point detection under reduced feedback. No GPU.

In [ ]:
import itertools
import numpy as np
import pandas as pd
from time import perf_counter
DOMAIN=np.array(list(itertools.product(range(3),repeat=4)),dtype=np.int8)
N=len(DOMAIN)
POWERS=np.array([27,9,3,1])
def index(x): return x@POWERS
PRIMITIVES={
 'left':np.roll(DOMAIN,-1,axis=1),'right':np.roll(DOMAIN,1,axis=1),
 'reverse':DOMAIN[:,::-1].copy(),'cycle':(DOMAIN+1)%3,'uncycle':(DOMAIN+2)%3,
 'swap01':np.array([1,0,2],dtype=np.int8)[DOMAIN],
 'swap12':np.array([0,2,1],dtype=np.int8)[DOMAIN],
 'swap_halves':DOMAIN[:,[2,3,0,1]]}
TRANSFORMS=[('before',p) for p in PRIMITIVES]+[('after',p) for p in PRIMITIVES]+[('guard',c) for c in range(3)]+[('identity',None)]
def transform(program,spec):
    kind,arg=spec
    if kind=='before': return program[index(PRIMITIVES[arg])]
    if kind=='after': return PRIMITIVES[arg][index(program)]
    if kind=='guard': return np.where(DOMAIN==arg,program,DOMAIN)
    if kind=='identity': return program.copy()
    raise ValueError(spec)
def candidates(source): return np.stack([transform(source,t) for t in TRANSFORMS])
def consistent(bank,ids,outputs): return np.all(bank[:,ids]==outputs[None],axis=(1,2))
def acquire_source(observed):
    bank=np.stack(list(PRIMITIVES.values()))
    alive=consistent(bank,np.arange(N),observed)
    assert alive.any()
    return bank[alive][0].copy()
TRAIN_BASES=['left','cycle','swap01','reverse']
TEST_BASES=['right','uncycle','swap12','swap_halves']
MACROS=[(i,j) for i in range(len(TRANSFORMS)) for j in range(len(TRANSFORMS))]
def macro_bank(source):
    return np.concatenate([candidates(source),np.stack([transform(transform(source,TRANSFORMS[i]),TRANSFORMS[j]) for i,j in MACROS])])
PAIR_FAMILIES={
 'matched':[(('guard',0),('after','cycle')),(('guard',1),('after','swap01')),(('guard',2),('after','reverse'))],
 'swapped':[(('guard',0),('after','swap01')),(('guard',1),('after','reverse')),(('guard',2),('after','cycle'))]}
def apply_pair(s,pair): return transform(transform(s,pair[0]),pair[1])
def learn_macros(family,seed=17,n=48):
    rng=np.random.default_rng(seed); m=len(TRANSFORMS); c=np.ones(m+m*m)
    for ep in range(n):
        s=acquire_source(PRIMITIVES[TRAIN_BASES[ep%len(TRAIN_BASES)]])
        pair=family[int(rng.integers(len(family)))]; y=apply_pair(s,pair)
        alive=consistent(macro_bank(s),np.arange(N),y)
        assert alive.any(); c+=alive/alive.sum()
    joint=c[m:].reshape(m,m); f=np.outer(joint.sum(1),joint.sum(0))/joint.sum()
    factor=np.concatenate([c[:m],f.ravel()])
    assert np.isclose(c.sum(),factor.sum())
    return c,factor


In [ ]:
# Same candidate bank as revision 3. All arms get exactly the same observations.
M=len(TRANSFORMS); D=M+M*M; U=np.ones(D)/D
HAZARD=.05; DECAY=.9; STICKY=.90

def norm(v): return v/v.sum(axis=-1,keepdims=True)
def factor(v):
    j=v[M:].reshape(M,M)
    return np.r_[v[:M],(np.outer(j.sum(1),j.sum(0))/j.sum()).ravel()]

class Fixed:
    def __init__(self,c): self.c=c.copy()
    def prior(self): return norm(self.c+U)
    def update(self,e): pass

class Forget(Fixed):
    def update(self,e): self.c=DECAY*self.c+e/e.sum()

class ChangePoint:
    # Full run-length support, no pruning; fractional-credit count update.
    def __init__(self,c): self.c=c[None].copy(); self.w=np.ones(1)
    def prior(self): return (1-HAZARD)*(self.w@norm(self.c+U))+HAZARD*U
    def update(self,e):
        p=norm(self.c+U); likelihood=p@e
        self.w=norm(np.r_[HAZARD*(U@e),(1-HAZARD)*self.w*likelihood])
        credit=e/e.sum(); self.c=np.vstack([credit,self.c+credit])

class Context:
    # A new context competes with previously learned contexts. No context IDs.
    def __init__(self,c,factorized=False):
        self.c=[c.copy()]; self.last=0; self.factorized=factorized; self.births=0; self.switches=0
    def components(self):
        k=len(self.c); w=np.full(k+1,(1-STICKY)/k); w[self.last]=STICKY
        p=np.stack([norm((factor(c) if self.factorized else c)+U) for c in self.c]+[U])
        return w,p
    def prior(self):
        w,p=self.components(); return w@p
    def update(self,e):
        w,p=self.components(); j=int(np.argmax(w*(p@e))); credit=e/e.sum()
        if j==len(self.c): self.c.append(credit); self.births+=1
        else: self.c[j]+=credit
        self.switches+=int(j!=self.last); self.last=j

def test_e4():
    e=np.zeros(D); e[0]=1; f=np.zeros(D); f[1]=1; c=48*e
    a=Context(c); before=a.prior().copy(); a.update(f)
    assert a.births==1 and a.last==1 and a.prior()[1]>before[1]
    a.update(e); assert a.last==0 and len(a.c)==2
    b=ChangePoint(c); b.update(f); assert np.isclose(b.w.sum(),1) and b.w[0]>.5
    for obj in (a,b,Forget(c),Fixed(c)):
        p=obj.prior(); assert p.shape==(D,) and np.isfinite(p).all() and np.isclose(p.sum(),1) and (p>0).all()
    print('E4 normalization, contradiction, new context and old-context return: PASS')

BANK={b:macro_bank(acquire_source(PRIMITIVES[b])) for b in TRAIN_BASES+TEST_BASES}
PAIR_IDX={k:[M+TRANSFORMS.index(i)*M+TRANSFORMS.index(j) for i,j in v] for k,v in PAIR_FAMILIES.items()}
def tasks(seed,schedule):
    rng=np.random.default_rng(seed)
    for family in schedule:
        b=rng.choice(TEST_BASES); bank=BANK[b]
        i=int(rng.choice(PAIR_IDX[family])) if family!='random' else int(rng.integers(D))
        target=bank[i]; perm=rng.permutation(N); train,test=perm[:8],perm[8:]
        alive=consistent(bank,train[:1],target[train[:1]])
        full=consistent(bank,np.arange(N),target)
        acc=np.all(bank[:,test]==target[None,test],axis=2).mean(1)
        yield alive,full,acc,rng.random(D)

def evaluate(seed,n=48):
    c=learn_macros(PAIR_FAMILIES['matched'],seed=101,n=48)[0]-1
    results=[]
    schedules={'stable':['matched']*60,'ABA':['matched']*20+['swapped']*20+['matched']*20,'random':['random']*60}
    for regime,schedule in schedules.items():
        for rep in range(n):
            arms={'uniform':Fixed(np.zeros(D)),'fixed factor':Fixed(factor(c)), 'fixed joint':Fixed(c), 'forget':Forget(c), 'change point':ChangePoint(c), 'context factor':Context(c,True),'context joint':Context(c)}
            for t,(alive,full,acc,tie) in enumerate(tasks(seed+rep,schedule)):
                for name,model in arms.items():
                    p=model.prior(); eligible=np.flatnonzero(alive)
                    pred=eligible[np.lexsort((tie[eligible],p[eligible]))[-1]]
                    results.append((regime,rep,t,name,acc[pred],getattr(model,'births',0),getattr(model,'switches',0)))
                    # Completed-task evidence is available only AFTER scoring.
                    model.update(full)
    return pd.DataFrame(results,columns=['regime','rep','task','arm','exact','births','switches'])

def summarize(df):
    a=df.copy(); a['phase']=a.task//20
    table=a.groupby(['regime','phase','arm']).exact.mean().unstack('arm')*100
    print(table.round(1).to_string())
    curve=a[a.regime=='ABA'].groupby(['task','arm']).exact.mean().unstack('arm')
    for name in ['forget','change point','context factor','context joint']:
        recovery=[]
        for start in (20,40):
            v=curve[name].to_numpy(); recovery.append(next((j-start for j in range(start,start+18) if (v[j:j+3]>=.9).all()),None))
        print(name,'recovery (offset, 3 consecutive >=90%):',recovery)
    delta=a.pivot(index=['regime','rep','task'],columns='arm',values='exact')
    for regime in ['stable','ABA','random']:
        d=(delta.loc[regime]['context joint']-delta.loc[regime]['change point']).groupby('rep').mean().to_numpy()*100
        boot=np.random.default_rng(99).choice(d,(2000,len(d)),replace=True).mean(1)
        print(regime,'context minus change point pp:',round(d.mean(),2),np.round(np.quantile(boot,[.025,.975]),2))
    print('context births/switches mean at final task:')
    print(a[(a.task==59)&a.arm.str.startswith('context')].groupby(['regime','arm'])[['births','switches']].mean().round(2).to_string())
    return table


In [ ]:
from threadpoolctl import threadpool_limits
import pandas as pd

def stress(seed=20270101,n=24):
    c=learn_macros(PAIR_FAMILIES['matched'],seed=101,n=48)[0]-1
    schedules={'irregular':['matched']*9+['swapped']*13+['matched']*5+['swapped']*9,
               'random':['random']*36}
    rows=[]
    for regime,schedule in schedules.items():
        for feedback in (1,4,81):
            for rep in range(n):
                rng=np.random.default_rng(seed+rep)
                arms={'uniform':Fixed(np.zeros(D)), 'fixed':Fixed(c), 'forget':Forget(c),
                      'change point':ChangePoint(c),'context':Context(c)}
                for t,family in enumerate(schedule):
                    bank=BANK[rng.choice(TEST_BASES)]
                    i=int(rng.choice(PAIR_IDX[family])) if family!='random' else int(rng.integers(D))
                    target=bank[i]; perm=rng.permutation(N); tie=rng.random(D)
                    obs=perm[:1]; held=perm[1:]; reveal=perm[:feedback]
                    alive=consistent(bank,obs,target[obs]); evidence=consistent(bank,reveal,target[reveal])
                    assert alive[i] and evidence[i] and np.all(~evidence|alive)
                    acc=np.all(bank[:,held]==target[None,held],axis=2).mean(1)
                    eligible=np.flatnonzero(alive)
                    for name,model in arms.items():
                        p=model.prior(); assert np.isfinite(p).all() and np.isclose(p.sum(),1)
                        pred=eligible[np.lexsort((tie[eligible],p[eligible]))[-1]]
                        rows.append((regime,feedback,rep,t,name,acc[pred]))
                        model.update(evidence)
    return pd.DataFrame(rows,columns=['regime','feedback','rep','task','arm','exact'])

def stress_report(df):
    p=df.pivot(index=['regime','feedback','rep','task'],columns='arm',values='exact')
    for (regime,feedback),g in p.groupby(level=['regime','feedback']):
        d=(g['context']-g['change point']).groupby(level='rep').mean().to_numpy()*100
        boot=np.random.default_rng(831).choice(d,(2000,len(d)),replace=True).mean(1)
        print(regime,feedback,'means', (100*g.mean()).round(2).to_dict(), 'context gap',round(d.mean(),2),np.quantile(boot,[.025,.975]).round(2))

with threadpool_limits(limits=1):
    df=stress(); stress_report(df)
df.to_csv('stress_v4.csv',index=False)

## Captured CPU results
Executed as a standalone Python script with identical definitions, single-thread BLAS. Re-running produces a CSV. No claims of notebook execution counts.
```text
irregular 1 means {'change point': 88.62, 'context': 88.26, 'fixed': 72.63, 'forget': 88.05, 'uniform': 52.83} context gap -0.36 [-1.88  1.11]
irregular 4 means {'change point': 92.07, 'context': 91.89, 'fixed': 72.63, 'forget': 90.52, 'uniform': 52.83} context gap -0.18 [-1.27  0.87]
irregular 81 means {'change point': 91.92, 'context': 91.96, 'fixed': 72.63, 'forget': 90.6, 'uniform': 52.83} context gap 0.03 [-1.05  1.1 ]
random 1 means {'change point': 56.21, 'context': 56.09, 'fixed': 57.59, 'forget': 55.53, 'uniform': 60.67} context gap -0.12 [-1.54  1.38]
random 4 means {'change point': 58.47, 'context': 56.77, 'fixed': 57.59, 'forget': 56.09, 'uniform': 60.67} context gap -1.7 [-3.42 -0.01]
random 81 means {'change point': 58.32, 'context': 57.02, 'fixed': 57.59, 'forget': 55.85, 'uniform': 60.67} context gap -1.3 [-3.    0.34]
```
**Decision:** no meaningful advantage across irregular shifts; random feedback sometimes degrades accuracy. This is stronger evidence against hard context allocation, not against program discovery. Do not scale this mechanism. The next serious test should be typed program synthesis with learned search and library compression, against retrieval and DreamCoder-style controls. No real-world generalization or SOTA claim. Remaining limitations: fixed bank, synthetic domain, complete source feedback, partial target evidence may remain ambiguous, no noisy observations or out-of-bank test yet.